# 03 · Two-stage anchor-based detector: Faster R-CNN

Faster R-CNN with a ResNet-50 FPN v2 backbone, COCO-pretrained (torchvision). Stage 1, the Region Proposal Network, scores anchors as object / background and refines them into proposals; stage 2 pools features for each proposal (RoIAlign) and predicts the class and a refined box. Experiments: the proposal / RoI budget for dense scenes and a GIoU box regression loss instead of smooth L1.

In [ ]:
import os, urllib.request
# helper modules shared by all notebooks (data split, metric, submission format, torchvision training)
REPO = "https://raw.githubusercontent.com/MakcDutch/AI-homework2/main/"
for f in ["common.py", "tv_train.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(REPO + f, f)

In [ ]:
import glob, time, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
import common as C
import tv_train as TV

ROOT = C.find_data_root()
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("work")
os.makedirs(WORK, exist_ok=True)
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## Protocol
Same frames, masking, input size and metric as `02_yolo`: every 5th train frame, ignored regions painted grey,
long side resized to 1024, evaluation on all val frames with `common.coco_evaluate`. Optimiser for both torchvision
models: SGD (momentum 0.9, weight decay 1e-4), 500-iteration warm-up, cosine decay, mixed precision, horizontal flip.
A val subset (every 5th frame) is scored after each epoch for the learning curve.

In [ ]:
TRAIN_STEP = 5
IMGSZ = 1024
EPOCHS = 6
BATCH = 4
LR = 0.005
WORKERS = 4
DATA_DIR = "/tmp/yolo_data"

EXPERIMENTS = {
    "frcnn_base": dict(),
    # dense scenes (~46 objects per frame, see 01_eda): more proposals kept and more RoIs sampled per image
    "frcnn_dense_rpn": dict(rpn_pre_nms_top_n_test=3000, rpn_post_nms_top_n_test=2000,
                            rpn_post_nms_top_n_train=3000, box_batch_size_per_image=1024),
    # loss modification: GIoU instead of smooth L1 for the second-stage box regression
    "frcnn_giou": dict(box_loss="giou"),
}
RUN = ["frcnn_base"]

In [ ]:
C.build_yolo_dataset(ROOT, DATA_DIR, train_step=TRAIN_STEP, val_step=5)
train_paths = sorted(glob.glob(f"{DATA_DIR}/images/train/*.jpg"))
val_frames = C.list_frames(ROOT, "val")
val_paths = val_frames["image"].tolist()
val_subset = val_frames[(val_frames["frame"] - val_frames.groupby("clip")["frame"].transform("min")) % 5 == 0]["image"].tolist()
test_paths = C.list_frames(ROOT, "test")["image"].tolist()
print(len(train_paths), "train |", len(val_paths), "val (", len(val_subset), "for curves ) |", len(test_paths), "test")

## Training, evaluation and submission

In [ ]:
def run_experiment(name, cfg):
    model = TV.build_faster_rcnn(IMGSZ, **cfg)
    t0 = time.time()
    history = TV.train_model(model, train_paths, IMGSZ, EPOCHS, BATCH, LR, f"{WORK}/{name}",
                             val_paths=val_subset, root=ROOT, workers=WORKERS)
    train_hours = (time.time() - t0) / 3600
    pd.DataFrame(history).to_csv(f"{WORK}/history_{name}.csv", index=False)

    val_preds, ms = TV.predict(model, val_paths, IMGSZ, workers=WORKERS)
    metrics, per_class = C.coco_evaluate(val_preds, ROOT, "val")
    test_preds, _ = TV.predict(model, test_paths, IMGSZ, workers=WORKERS)

    C.save_preds(val_preds, f"{WORK}/val_preds_{name}.pkl")
    C.save_preds(test_preds, f"{WORK}/test_preds_{name}.pkl")
    C.make_submission(test_preds, ROOT, f"{WORK}/sub_{name}.csv")
    pd.Series(per_class, name=name).to_csv(f"{WORK}/per_class_{name}.csv")
    row = dict(exp=name, family="two-stage anchor-based", imgsz=IMGSZ, epochs=EPOCHS,
               train_hours=round(train_hours, 2), ms_per_img=round(ms, 1), **metrics)
    C.results_table(f"{WORK}/results.csv", row)
    del model; torch.cuda.empty_cache()
    return row


for name in RUN:
    print(run_experiment(name, EXPERIMENTS[name]))

## Results

In [ ]:
display(C.results_table(f"{WORK}/results.csv"))
per_class = pd.concat([pd.read_csv(f"{WORK}/per_class_{n}.csv", index_col=0) for n in RUN], axis=1)
display(per_class.round(3))

fig, ax = plt.subplots(figsize=(7, 3.5))
for name in RUN:
    h = pd.read_csv(f"{WORK}/history_{name}.csv")
    ax.plot(h["epoch"], h["val_mAP"], marker="o", label=name)
ax.set_xlabel("epoch"); ax.set_ylabel("val-subset mAP"); ax.legend()
plt.tight_layout(); plt.savefig(f"{WORK}/curves.png", dpi=120); plt.show()

## What each stage contributes
Recall of the RPN proposals vs recall of the final detections on 150 val frames. The gap between the two columns
is what the second stage loses (or keeps) when it classifies and suppresses proposals.

In [ ]:
name = RUN[-1]
model = TV.build_faster_rcnn(IMGSZ, **EXPERIMENTS[name])
model.load_state_dict(torch.load(f"{WORK}/{name}/last.pt", map_location="cpu"))
sample = val_paths[:: max(1, len(val_paths) // 150)]
display(TV.rpn_vs_final_recall(model, ROOT, sample, IMGSZ))
del model; torch.cuda.empty_cache()

## Failure cases
Ground truth (class colours), ignored regions (red hatch), predictions with score ≥ 0.3 (white dashed).

In [ ]:
preds = C.load_preds(f"{WORK}/val_preds_{RUN[-1]}.pkl")
C.show_failures(preds, ROOT, f"{WORK}/failures_{RUN[-1]}.png")